# 1 - Custom pipeline walkthrough

Runs the **custom** Sentinel-1 ocean-current retrieval for one scene and shows the field after each step. The Doppler centroid is estimated for each TOPS burst with GAMMA, the geometry is removed from the precise orbit, and the sideband, descalloping, mispointing, Stokes and wave (Mouche) corrections are applied. Instrument steps are shown in Hz; velocity and current in m/s, on the native range/azimuth grid until the final gridding.

In [ ]:
import os, sys
from pathlib import Path
# resolve repo root whether launched from notebooks/ or the repo root
REPO = os.getcwd()
if not os.path.exists(os.path.join(REPO, 'scripts', 'sentinel_1')):
    REPO = os.path.abspath(os.path.join(REPO, '..'))
os.chdir(REPO); sys.path.insert(0, REPO)
import numpy as np
import matplotlib.pyplot as plt
from scripts.run_pipeline import scene_paths, CUSTOM_CFG
from scripts.sentinel_1.grid_merge import merge_burst_grids
print('repo root:', REPO)

## Run the pipeline
Calls GAMMA (import, precise orbit, deramping, Doppler) and the correction chain.

In [ ]:
from scripts.sentinel_1.pipeline import run_gamma_pipeline_from_safe
paths = scene_paths('data', scene='scene1', subswath='iw1', pol='vv')
R = run_gamma_pipeline_from_safe(**paths, keep_products=True,
        products_dir='data/sentinel-1/gamma_iw1', **CUSTOM_CFG)
BURSTS = [R]
print('fields:', sorted(k for k in R if hasattr(R[k], 'shape')))

## Step-by-step fields

In [ ]:
def show(field, title, unit, cmap='RdBu_r'):
    f = np.asarray(field, float); v = f[np.isfinite(f)]
    lo, hi = np.nanpercentile(v, 2), np.nanpercentile(v, 98)
    if lo >= 0 or hi <= 0:
        kw = dict(cmap='viridis', vmin=lo, vmax=hi)
    else:
        m = max(abs(lo), abs(hi)); kw = dict(cmap=cmap, vmin=-m, vmax=m)
    fig, ax = plt.subplots(figsize=(5, 5.4), constrained_layout=True)
    im = ax.imshow(f, origin='lower', aspect='auto', interpolation='nearest', **kw)
    ax.set_title(title); ax.set_xlabel('range [block]'); ax.set_ylabel('azimuth [block]')
    fig.colorbar(im, ax=ax, shrink=0.9, label=unit); plt.show()

### Doppler domain (Hz)

In [ ]:
show(R['f_dc'], 'Measured Doppler centroid', 'Hz', cmap='viridis')
show(R['f_dca_pre_descallop'], 'Anomaly post geometry + sideband', 'Hz')
show(R['f_dca'], 'Anomaly post descalloping', 'Hz')
show(R['f_miss_ocn'], 'Mispointing Doppler correction', 'Hz', cmap='viridis')
show(R['f_dca'] - R['f_miss_ocn'], 'Anomaly post mispointing', 'Hz')

### Velocity domain (m/s)

In [ ]:
show(-R['v_r'] + R['v_miss_ocn'], 'Radial velocity (instrument-corrected)', 'm/s')
show(R['v_wave'] + R['v_stokes'], 'Wave + Stokes correction', 'm/s')
show(R['v_current_ocn'], 'Surface radial current', 'm/s')

## Final gridded current

In [ ]:
glat, glon, g = merge_burst_grids(BURSTS, variable='v_current_ocn', overlap='average', resolution_deg=0.01)
v = g[np.isfinite(g)]; m = np.nanpercentile(np.abs(v), 98)
fig, ax = plt.subplots(figsize=(6, 6.4), constrained_layout=True)
im = ax.pcolormesh(glon, glat, g, cmap='RdBu_r', vmin=-m, vmax=m, shading='auto')
ax.set_title('Custom pipeline: surface radial current'); ax.set_xlabel('lon'); ax.set_ylabel('lat')
fig.colorbar(im, ax=ax, shrink=0.9, label='radial current [m/s]'); plt.show()
print('mean %.3f  range [%.3f, %.3f] m/s' % (v.mean(), v.min(), v.max()))

> Command-line equivalent: `python scripts/run_pipeline.py`